In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import os

def crawl_fund_redemption_list(exchange, date=None):
    """
    爬取指定交易所的基金申赎清单
    :param exchange: 'sse'（上交所）或'szse'（深交所）
    :param date: 日期，格式如'20250923'，默认今天
    :return: 基金申赎清单数据框
    """
    if date is None:
        date = time.strftime("%Y%m%d")
    
    # 更新URL - 尝试最新的交易所URL结构
    if exchange == 'sse':
        # 上交所ETF申赎清单页面可能的URL
        url = f"http://www.sse.com.cn/assortment/fund/etf/redemption/detail/?date={date}"
    elif exchange == 'szse':
        # 深交所ETF申赎清单页面可能的URL
        url = f"http://www.szse.cn/market/product/etf/list/detail/?date={date}"
    else:
        raise ValueError("交易所类型错误，应为'sse'或'szse'")
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36",
        "Referer": f"http://www.{exchange}.com.cn/",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8",
        "Accept-Language": "zh-CN,zh;q=0.9"
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        if exchange == 'sse':
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # 尝试多种可能的表格结构
            possible_tables = [
                soup.find('table', class_='table'),
                soup.find('table', id='dataTable'),
                soup.find('table', class_='table-border')
            ]
            
            table = None
            for t in possible_tables:
                if t:
                    table = t
                    break
                    
            if not table:
                print(f"在上交所页面未找到数据表格: {url}")
                return pd.DataFrame()
                
            # 提取表头和数据
            headers = []
            for th in table.find_all('th')[:6]:
                headers.append(th.text.strip())
                
            data = []
            for row in table.find_all('tr')[1:]:
                cols = row.find_all('td')[:6]
                if len(cols) >= 6:
                    row_data = {headers[i]: cols[i].text.strip() for i in range(6)}
                    data.append(row_data)
                    
            df = pd.DataFrame(data)
            if not df.empty:
                df.columns = ["基金代码", "基金名称", "申赎代码", "申赎单位", "申购清单", "赎回清单"]
                
        elif exchange == 'szse':
            # 尝试解析HTML表格
            soup = BeautifulSoup(response.text, 'html.parser')
            
            possible_tables = [
                soup.find('table', class_='table'),
                soup.find('table', id='redemptionList'),
                soup.find('table', class_='table-data')
            ]
            
            table = None
            for t in possible_tables:
                if t:
                    table = t
                    break
                    
            if not table:
                # 尝试JSON解析作为备选方案
                try:
                    json_data = response.json()
                    items = json_data.get('data', {}).get('items', [])
                    if not items:
                        items = json_data.get('list', [])
                        
                    data = []
                    for item in items:
                        data.append({
                            "基金代码": item.get('fundCode', item.get('code', '')),
                            "基金名称": item.get('fundName', item.get('name', '')),
                            "申赎代码": item.get('redemptionCode', item.get('rCode', '')),
                            "申赎单位": item.get('redemptionUnit', item.get('unit', '')),
                            "申购清单": item.get('purchaseList', item.get('purchase', '')),
                            "赎回清单": item.get('redemptionList', item.get('redemption', ''))
                        })
                    df = pd.DataFrame(data)
                except:
                    print(f"在深交所页面未找到数据: {url}")
                    return pd.DataFrame()
            else:
                # 解析HTML表格
                headers = []
                for th in table.find_all('th')[:6]:
                    headers.append(th.text.strip())
                    
                data = []
                for row in table.find_all('tr')[1:]:
                    cols = row.find_all('td')[:6]
                    if len(cols) >= 6:
                        row_data = {headers[i]: cols[i].text.strip() for i in range(6)}
                        data.append(row_data)
                        
                df = pd.DataFrame(data)
                if not df.empty:
                    df.columns = ["基金代码", "基金名称", "申赎代码", "申赎单位", "申购清单", "赎回清单"]
            
        return df
        
    except requests.exceptions.HTTPError as e:
        print(f"HTTP错误: {e}")
        print(f"建议手动检查URL是否正确: {url}")
        return pd.DataFrame()
    except requests.exceptions.RequestException as e:
        print(f"请求错误: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"解析错误: {e}")
        return pd.DataFrame()

# 爬取上交所和深交所数据
print("开始爬取上交所基金申赎清单...")
sse_df = crawl_fund_redemption_list('sse')
print(f"上交所数据爬取完成，共{len(sse_df)}条记录")

print("开始爬取深交所基金申赎清单...")
szse_df = crawl_fund_redemption_list('szse')
print(f"深交所数据爬取完成，共{len(szse_df)}条记录")

# 保存为Excel文件
output_path = 'fund_redemption_list.xlsx'
with pd.ExcelWriter(output_path) as writer:
    sse_df.to_excel(writer, sheet_name='上交所', index=False)
    szse_df.to_excel(writer, sheet_name='深交所', index=False)

print(f"数据已保存至 {os.path.abspath(output_path)}")

# 如果爬取失败，提供手动获取建议
if len(sse_df) == 0 and len(szse_df) == 0:
    print("\n爬取失败，建议手动访问以下网址获取数据：")
    print(f"上交所基金申赎清单：http://www.sse.com.cn/assortment/fund/etf/redemption/")
    print(f"深交所基金申赎清单：http://www.szse.cn/market/product/etf/list/")
    

开始爬取上交所基金申赎清单...
HTTP错误: 404 Client Error: Not Found for url: http://www.sse.com.cn/assortment/fund/etf/redemption/detail/?date=20250923
建议手动检查URL是否正确: http://www.sse.com.cn/assortment/fund/etf/redemption/detail/?date=20250923
上交所数据爬取完成，共0条记录
开始爬取深交所基金申赎清单...
HTTP错误: 404 Client Error: Not Found for url: http://www.szse.cn/market/product/etf/list/detail/?date=20250923
建议手动检查URL是否正确: http://www.szse.cn/market/product/etf/list/detail/?date=20250923
深交所数据爬取完成，共0条记录
数据已保存至 C:\Users\86157\量化交易习题\fund_redemption_list.xlsx

爬取失败，建议手动访问以下网址获取数据：
上交所基金申赎清单：http://www.sse.com.cn/assortment/fund/etf/redemption/
深交所基金申赎清单：http://www.szse.cn/market/product/etf/list/
